<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-06a-kestrel-hourly-forecaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 6, track (a) — Kestrel hourly forecaster
**Course 1: Hands-On Deep Learning with Python — Chapter 6: Sequence models**

**Problem brief (Leo Farkas, Kestrel Logistics):** "We over- and under-staff our main hub
because our demand forecast is a 7-day moving average. Give us an hourly throughput
forecast 24 hours ahead." Target: beat the moving-average baseline MAE by ≥ 20%.

*(Choose this track OR track (b), `lab-06b-fernwood-article-router.ipynb` — not both.)*

**What you'll submit:** an LSTM forecaster with a proper temporal split, a rolling-origin
backtest, the MAE comparison against the baseline, and an error analysis.

## 1. Load the data (with offline fallback)

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd

np.random.seed(0)

def load_traffic_data():
    try:
        # fetch with a bounded timeout first - pd.read_csv(url) has no timeout of its own
        # and can hang the whole cell indefinitely on a stalled connection
        req = urllib.request.Request(
            'https://archive.ics.uci.edu/ml/machine-learning-databases/00492/'
            'Metro_Interstate_Traffic_Volume.csv.gz',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_csv(io.BytesIO(raw), compression='gzip', parse_dates=['date_time'])
        print('Loaded the real Metro Interstate Traffic Volume dataset:', df.shape)
        return df.sort_values('date_time').reset_index(drop=True)
    except Exception as e:
        print(f'Offline fallback engaged ({e}) — generating a synthetic hourly series with a\n'
              'daily + weekly seasonal pattern, similar in spirit to real hub throughput.')
        hours = pd.date_range('2023-01-01', periods=24 * 400, freq='h')
        t = np.arange(len(hours))
        daily = 2500 + 1800 * np.sin(2 * np.pi * (t % 24) / 24 - 1.2)
        weekly = 300 * np.sin(2 * np.pi * (t % (24 * 7)) / (24 * 7))
        noise = np.random.normal(0, 200, len(t))
        volume = np.clip(daily + weekly + noise, 0, None)
        return pd.DataFrame({'date_time': hours, 'traffic_volume': volume})

df = load_traffic_data()
series = df['traffic_volume'].to_numpy(dtype=np.float32)
print('series length:', len(series))

## 2. Windowing + a proper temporal split

In [ ]:
import torch

WINDOW = 48   # look back 48 hours
HORIZON = 24  # predict 24 hours ahead

mean_, std_ = series.mean(), series.std()
series_norm = (series - mean_) / std_

def make_windows(arr, window, horizon):
    X, y = [], []
    for i in range(len(arr) - window - horizon + 1):
        X.append(arr[i:i + window])
        y.append(arr[i + window + horizon - 1])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

X_all, y_all = make_windows(series_norm, WINDOW, HORIZON)

# temporal split — the val set is strictly AFTER the train set, never shuffled
split = int(len(X_all) * 0.8)
X_train, y_train = X_all[:split], y_all[:split]
X_val, y_val = X_all[split:], y_all[split:]
print('train:', X_train.shape, 'val:', X_val.shape)

## 3. The baseline: 7-day (168h) moving average

In [ ]:
def moving_average_forecast(arr, window, horizon, ma_window=168):
    preds = []
    for i in range(len(arr) - window - horizon + 1):
        history_end = i + window
        lookback = arr[max(0, history_end - ma_window):history_end]
        preds.append(lookback.mean())
    return np.array(preds, dtype=np.float32)

baseline_preds_norm = moving_average_forecast(series_norm, WINDOW, HORIZON)
baseline_preds = baseline_preds_norm[split:] * std_ + mean_
y_val_real = y_val * std_ + mean_
baseline_mae = np.mean(np.abs(baseline_preds - y_val_real))
print(f'Baseline (7-day moving average) MAE: {baseline_mae:.1f}')
print(f'Target: an LSTM MAE at or below {baseline_mae * 0.8:.1f} (20% better)')

## 4. The LSTM forecaster

In [ ]:
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'

class LSTMForecaster(nn.Module):
    def __init__(self, hidden=64, n_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden, num_layers=n_layers, batch_first=True)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        # x: (batch, window) -> (batch, window, 1)
        out, (h_n, c_n) = self.lstm(x.unsqueeze(-1))
        last_hidden = h_n[-1]  # (batch, hidden) — the final layer's final hidden state
        return self.head(last_hidden).squeeze(-1)

train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
model = LSTMForecaster().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(15):
    model.train()
    epoch_loss, nb = 0.0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item(); nb += 1
    if epoch % 3 == 0 or epoch == 14:
        print(f'epoch {epoch}: train MSE (normalized) = {epoch_loss / nb:.4f}')

## 5. Rolling-origin backtest vs. the baseline

In [ ]:
model.eval()
with torch.no_grad():
    val_preds_norm = model(torch.tensor(X_val).to(device)).cpu().numpy()
val_preds = val_preds_norm * std_ + mean_
lstm_mae = np.mean(np.abs(val_preds - y_val_real))

print(f'Baseline MAE: {baseline_mae:.1f}')
print(f'LSTM MAE:     {lstm_mae:.1f}')
improvement = (baseline_mae - lstm_mae) / baseline_mae * 100
print(f'Improvement: {improvement:.1f}%  (target: >= 20%)')

In [ ]:
import matplotlib.pyplot as plt

n_show = 200
plt.figure(figsize=(10, 3))
plt.plot(y_val_real[:n_show], label='actual', alpha=0.8)
plt.plot(baseline_preds[:n_show], label='moving-average baseline', alpha=0.7)
plt.plot(val_preds[:n_show], label='LSTM', alpha=0.7)
plt.legend(); plt.title('Kestrel hub throughput — actual vs. forecasts'); plt.xlabel('validation hour')
plt.show()

# This chart doubles as the seed of the drift monitor Chapter 10 builds in full:
errors = np.abs(val_preds - y_val_real)
plt.figure(figsize=(10, 2.5))
plt.plot(errors)
plt.axhline(errors.mean(), color='r', linestyle='--', label='mean abs error')
plt.title('Forecast error over time — the seed of a drift monitor'); plt.legend()
plt.show()

## 6. Error analysis (fill in)
Where does the LSTM do worse than the baseline, if anywhere (e.g. sudden spikes, holidays,
the start of the validation window)? What would you tell Kestrel about when to trust this
forecast less?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 6: Sequence models*